# 🦋 City Nature Challenge: SF Bay Area - Data Ingest
### Powered by Elasticsearch Serverless + Mistral

The [City Nature Challenge](https://www.inaturalist.org/projects/city-nature-challenge-2026-san-francisco-bay-area) is an annual springtime bioblitz that started in San Francisco and Los Angeles in 2016. Every year, Bay Area residents photograph and record every wild plant, animal, bug, and fungus they can find: the 2026 Bay Area event logged **~65,000 observations**. Every record comes from [iNaturalist](https://www.inaturalist.org/api) with a **photo** (and sometimes a **sound recording**), a crowd-sourced species ID, a location, and a timestamp.

**What this notebook does:**
1. Pulls City Nature Challenge observations from the iNaturalist API - **free, no key** (cursor-paged, so you can grab them all)
2. Optionally narrows them to **San Francisco County** only
3. Normalises each observation (taxon, location → `geo_point`, photo + sound URLs) and bulk-indexes it into `sf_city_nature_challenge`
4. **(Optional)** uses a vision-capable **Mistral** model to identify a species from its photo

This is the most *multimodal* dataset in the repo: images for Mistral vision, audio for Voxtral, and rich text for embeddings - all geo-tagged.

---
> **Data source:** iNaturalist, *City Nature Challenge: San Francisco Bay Area* (collection project). Observations are CC-licensed by their individual observers - the per-record `license` is indexed. Be gentle on the public API: the notebook paces itself.

## 0. Install dependencies

In [ ]:
!pip install elasticsearch requests --quiet

In [ ]:
import time, requests
from elasticsearch import Elasticsearch, helpers

print('✅ Imports ready')

## 1. Connect to Elastic Serverless
> **Elastic Cloud Console → Your Project → Connection Details**

In [ ]:
# 🔑 Fill these in
ELASTIC_ENDPOINT = 'https://your-project.es.region.aws.elastic.cloud'
ELASTIC_API_KEY  = 'your-elastic-api-key'

es = Elasticsearch(ELASTIC_ENDPOINT, api_key=ELASTIC_API_KEY)
print(f'✅ Connected to Elasticsearch {es.info()["version"]["number"]}')

## 2. Pull the observations from iNaturalist
The API caps any single query at 10,000 results, so we page with an **`id_above` cursor** - ask for the next batch of IDs greater than the last one we saw - until we hit `SAMPLE_SIZE` or run out.

Fetching 200 observations per page with a 1-second pause takes a minute or two for 10,000 rows, and the counter updates in place so it can look stuck when it isn't. If you interrupt the cell, the observations fetched so far are kept.

The Bay Area project covers nine counties. Set `SF_ONLY = True` to keep just San Francisco (iNaturalist place `854`).

In [ ]:
# year → iNaturalist project_id  (umbrella across all years: 133934)
PROJECTS = {2017: 11013, 2018: 16036, 2019: 29624, 2020: 62485, 2021: 82726,
            2022: 117403, 2023: 146321, 2024: 181463, 2025: 226032, 2026: 263576}
YEAR        = 2026
PROJECT_ID  = PROJECTS[YEAR]
SF_ONLY     = True      # False = the whole nine-county Bay Area
SAMPLE_SIZE = 10000     # raise toward the full count; lower for a quick run
PER_PAGE    = 200
UA = {'User-Agent': 'sf-hacknight-inaturalist/1.0 (Elastic x Mistral; non-commercial)'}

def fetch_observations(project_id, limit):
    url = 'https://api.inaturalist.org/v1/observations'
    out, last_id = [], 0
    try:
        while len(out) < limit:
            params = {'project_id': project_id, 'per_page': PER_PAGE,
                      'order_by': 'id', 'order': 'asc', 'id_above': last_id}
            if SF_ONLY:
                params['place_id'] = 854   # San Francisco County
            r = requests.get(url, params=params, headers=UA, timeout=60)
            r.raise_for_status()
            batch = r.json()['results']
            if not batch:
                break
            out.extend(batch)
            last_id = batch[-1]['id']
            print(f'  fetched {len(out)} observations...', end='\r', flush=True)
            time.sleep(1.0)   # be polite to the public API
    except KeyboardInterrupt:
        # Stopped early (interrupt button): keep what we have so the rest of the notebook still runs
        print(f'\n⏹️  Interrupted - keeping the {len(out)} observations fetched so far')
    return out[:limit]

raw = fetch_observations(PROJECT_ID, SAMPLE_SIZE)
print(f'\n✅ Fetched {len(raw)} observations for CNC {YEAR}')

## 3. Create index with explicit mappings

In [ ]:
INDEX = 'sf_city_nature_challenge'

mapping = {'mappings': {'properties': {
    'observation_id': {'type': 'keyword'},
    'observed_on':    {'type': 'date'},
    'year':           {'type': 'integer'},
    'quality_grade':  {'type': 'keyword'},   # research / needs_id / casual
    'license':        {'type': 'keyword'},
    'scientific_name':{'type': 'keyword'},
    'common_name':    {'type': 'text', 'fields': {'keyword': {'type': 'keyword'}}},
    'rank':           {'type': 'keyword'},   # species, genus, family, ...
    'iconic_taxon':   {'type': 'keyword'},   # Plantae, Aves, Insecta, Mammalia, Fungi, ...
    'native':         {'type': 'boolean'},   # established natively in this place (when known)
    'introduced':     {'type': 'boolean'},
    'description':    {'type': 'text'},
    'place_guess':    {'type': 'text'},
    'location':       {'type': 'geo_point'},
    'observer':       {'type': 'keyword'},
    'num_photos':     {'type': 'integer'},
    'num_sounds':     {'type': 'integer'},
    'photo_url':      {'type': 'keyword'},   # medium-size image (for Mistral vision)
    'sound_url':      {'type': 'keyword'},   # audio file (for Voxtral)
    'uri':            {'type': 'keyword'},   # link to the observation on iNaturalist
}}}

if es.indices.exists(index=INDEX):
    es.indices.delete(index=INDEX)
    print(f'🗑️  Deleted existing index: {INDEX}')
es.indices.create(index=INDEX, body=mapping)
print(f'✅ Created index: {INDEX}')

## 4. Enrich and ingest

In [ ]:
def enrich(r):
    t = r.get('taxon') or {}
    photos = r.get('photos') or []
    sounds = r.get('sounds') or []
    doc = {
        'observation_id': str(r['id']),
        'observed_on':    r.get('observed_on'),
        'year':           YEAR,
        'quality_grade':  r.get('quality_grade'),
        'license':        r.get('license_code'),
        'scientific_name':t.get('name'),
        'common_name':    t.get('preferred_common_name'),
        'rank':           t.get('rank'),
        'iconic_taxon':   t.get('iconic_taxon_name'),
        'native':         t.get('native'),
        'introduced':     t.get('introduced'),
        'description':    r.get('description'),
        'place_guess':    r.get('place_guess'),
        'observer':       (r.get('user') or {}).get('login'),
        'num_photos':     len(photos),
        'num_sounds':     len(sounds),
        'uri':            r.get('uri'),
    }
    # "lat,lon" string → geo_point
    loc = r.get('location')
    if loc and ',' in loc:
        try:
            lat, lon = (float(x) for x in loc.split(','))
            doc['location'] = {'lat': lat, 'lon': lon}
        except ValueError:
            pass
    # medium-size photo (swap the 'square' thumbnail) for Mistral vision
    if photos and photos[0].get('url'):
        doc['photo_url'] = photos[0]['url'].replace('square', 'medium')
    if sounds and sounds[0].get('file_url'):
        doc['sound_url'] = sounds[0]['file_url']
    return {k: v for k, v in doc.items() if v not in (None, '')}

docs = [enrich(r) for r in raw]
success, errors = helpers.bulk(es, [{'_index': INDEX, '_source': d} for d in docs], raise_on_error=False)
es.indices.refresh(index=INDEX)
print(f'✅ Indexed {success} documents into {INDEX} ({len(errors)} errors)')

## 5. Verify - what did San Francisco find?

In [ ]:
resp = es.search(index=INDEX, body={'size': 0, 'aggs': {
    'by_kingdom':  {'terms': {'field': 'iconic_taxon', 'size': 12}},
    'top_species': {'terms': {'field': 'common_name.keyword', 'size': 10}},
    'by_quality':  {'terms': {'field': 'quality_grade', 'size': 5}},
}})
print(f"Total observations: {resp['hits']['total']['value']}\n")
print('By group (iconic taxon):')
for b in resp['aggregations']['by_kingdom']['buckets']:
    print(f"  {b['key']:14s} {b['doc_count']}")
print('\nMost-observed species:')
for b in resp['aggregations']['top_species']['buckets']:
    print(f"  {b['key']:30s} {b['doc_count']}")
print('\nBy quality grade:')
for b in resp['aggregations']['by_quality']['buckets']:
    print(f"  {b['key']:12s} {b['doc_count']}")

In [ ]:
# Observations with sound recordings (birds, frogs, insects) — the Voxtral candidates
resp = es.search(index=INDEX, body={'size': 5, 'query': {'range': {'num_sounds': {'gte': 1}}},
                                            '_source': ['common_name', 'scientific_name', 'sound_url']})
print(f"Observations with audio: {resp['hits']['total']['value']}")
for h in resp['hits']['hits']:
    s = h['_source']
    print(f"  {s.get('common_name') or s.get('scientific_name')}: {s.get('sound_url')}")

## 6. (Optional) Identify a species from its photo with Mistral 👀
Send an observation photo to a vision-capable Mistral model and compare its guess with the crowd ID.

> Needs your **Mistral API key**.

In [ ]:
MISTRAL_API_KEY = '<MISTRAL_API_KEY>'  # 🔑 paste your Mistral key

def identify(photo_url):
    r = requests.post('https://api.mistral.ai/v1/chat/completions',
        headers={'Authorization': f'Bearer {MISTRAL_API_KEY}', 'Content-Type': 'application/json'},
        json={'model': 'mistral-large-4', 'messages': [{'role': 'user', 'content': [
            {'type': 'text', 'text': 'This photo was taken in San Francisco. What species is this? '
                                     'Answer with the common and scientific name only.'},
            {'type': 'image_url', 'image_url': photo_url},
        ]}]},
        timeout=60)
    r.raise_for_status()
    return r.json()['choices'][0]['message']['content']

resp = es.search(index=INDEX, body={'size': 3, 'query': {'bool': {'filter': [
    {'term': {'quality_grade': 'research'}}, {'exists': {'field': 'photo_url'}}]}}})
for h in resp['hits']['hits']:
    s = h['_source']
    print(f"Crowd ID: {s.get('common_name')} ({s.get('scientific_name')})")
    print(f"Mistral:  {identify(s['photo_url'])}\n")

## 7. Going further with Mistral

- **Pocket naturalist:** snap a photo, identify it with Mistral vision, then show where else in SF it was seen.
- **Native plant finder:** filter on `native`, aggregate by neighborhood (geo), and have Mistral suggest what to plant in a backyard or sidewalk garden.
- **Bird calls:** transcribe or describe the audio clips with **Voxtral**.
- **Semantic search** over observation descriptions with Mistral embeddings (see [using_mistral_in_elasticsearch.md](using_mistral_in_elasticsearch.md)).

See [mistral_guide.md](mistral_guide.md) for calling Mistral directly.

---
## ✅ Index loaded
`sf_city_nature_challenge` captures San Francisco's wild side - geo-tagged plants, birds, bugs and more, each with a photo (and sometimes audio).

Combine it with [sf_street_trees](sf_street_trees.ipynb) and [sf_parks_and_public_art](sf_parks_and_public_art.ipynb) for a map of the city's nature.